In [1]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [2]:
import nltk
nltk.download('punkt'),  nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


(True, True)

# Train

In [3]:
import pandas as pd
import numpy as np
import joblib
import json
from itertools import chain
import click
import os

from ruamel.yaml import YAML

# Загрузка данных

In [4]:
from src.train import load_external_data, enc_classes

# получаем данные митра
conf = YAML().load(open('params.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))
conf_bert_ttp['nn'] = conf_bert_ttp['nn_ttp']
conf_bert_ttp['nn_bert'] = conf_bert_ttp['nn_bert_ttp']


/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
data = pd.read_csv(conf['prep_text']['prep_fn'])
data.target_ttp = data.target_ttp.map(lambda x: eval(x))
data.ttp = data.ttp.map(lambda x: eval(x))
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb = joblib.load(conf['prep_text']['mlb_fn'])

/opt/conda/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultiLabelBinarizer from version 1.6.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
import torch

model_bert_ttp = torch.load(conf_bert_ttp['nn_bert']['model_fn'])

model_bert = torch.load(conf_bert['nn_bert']['model_fn'])
    
thresh_ttp_l = joblib.load(conf_bert_ttp['nn_bert']['thresh_l_fn'])
thresh_l = joblib.load(conf_bert['nn_bert']['thresh_l_fn'])




/tmp/ipykernel_2207/2052135201.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_bert_ttp = torch.load(conf_bert_ttp['nn_bert']['model_fn'])
/tmp/ipykernel_2207/2052

In [9]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb = joblib.load(conf['prep_text']['mlb_fn'])

## nttp pred

In [10]:
from src.predict import predict_bert



pred_df = predict_bert(data[['sentence', 'labels', 'ttp', 'split', 'target_ttp']].assign(target=1), model_bert, thresh_l, conf_bert, suf='labels')
pred_df.head(2)

,sentence,labels,ttp,split,target_ttp,target,proba_labels,pred_labels
0,Adversaries may inject malicious code into pro...,"['defense-evasion', 'privilege-escalation']",[rare],tr,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.9855329394340515, 0.9938104748725891, 0.046...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,"Before creating a window, graphical Windows-ba...","['defense-evasion', 'privilege-escalation']",[rare],ts,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.04536919295787811, 0.2358996868133545, 0.40...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [11]:

pred_true_df = pred_df.copy().assign(target=data.labels.map(lambda x: mlb.transform([eval(x)])[0]))

In [12]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

(0.7059942911512845, 0.6720454775643292)

## ttp pred

In [13]:
pred_df = predict_bert(pred_df, model_bert_ttp, thresh_ttp_l, conf_bert_ttp, suf='ttp')
pred_df.head(2)

,sentence,labels,ttp,split,target_ttp,target,proba_labels,pred_labels,proba_ttp,pred_ttp
0,Adversaries may inject malicious code into pro...,"['defense-evasion', 'privilege-escalation']",[rare],tr,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.9855329394340515, 0.9938104748725891, 0.046...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.358859123001821e-07, 2.0291264490879257e-07...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"Before creating a window, graphical Windows-ba...","['defense-evasion', 'privilege-escalation']",[rare],ts,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.04536919295787811, 0.2358996868133545, 0.40...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.275171371162287e-06, 2.4010474589886144e-05...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
pred_true_df = pred_df.copy().assign(target=data.target_ttp)

In [15]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

(0.6545271683032051, 0.6784715015698937)

# Обучение

In [16]:
df = pred_df

In [ ]:
%%time
from src.funcs import set_seed
# set_seed(conf['seed'])

from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from src.meta import BFSThres, fake_splitter, get_toolbox, get_gen_proba
from src.funcs import get_pred_thresh

from sklearn.metrics import average_precision_score, f1_score
from deap import tools
from deap import algorithms

# model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
#                             class_weight='balanced', penalty=None)

model = DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=conf['seed'])

average_precision_score_l = []
average_precision_score_bert_l = []
f1_score_l = []
f1_score_bert_l = []

average_precision_score_tr_l = []
average_precision_score_tr_bert_l = []

average_precision_score_val_l = []
average_precision_score_val_bert_l = []
f1_score_val_l = []
f1_score_bert_val_l = []
f_l = []
feat_l = []

N_GEN = 100

from time import time

st = time()
tak_col = 'proba_labels' # 'proba_labels'
i_l = []
for i in tqdm(range(len(mlb_ttp.classes_))):
    
    X = df.apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1)

    X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    
    y = df['target_ttp'].map(lambda x: x[i])
    
    selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.1, metric_sign=1)
    selector.fit(X, y, verbose=True)
    
    filt_cols = np.array(selector.mask)
    filt_cols = X.columns[filt_cols]
    if not 'ttp_i' in filt_cols:
        # import pdb;pdb.set_trace()
        filt_cols = filt_cols.tolist() + ['ttp_i']
    if len(filt_cols)==1:
        filt_cols = ['ttp_i']
        continue
    i_l.append(i)
    X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_tr = df.query('split=="tr"')['target_ttp'].map(lambda x: x[i])
    
    X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_val = df.query('split=="val"')['target_ttp'].map(lambda x: x[i])
    
    X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_ts = df.query('split=="ts"')['target_ttp'].map(lambda x: x[i])


    feat_l.append(filt_cols)
    
    N = len(filt_cols)
    
    toolbox = get_toolbox(N=N)

    def evalSymbReg(individual, points, y_true):
        # Transform the tree expression in a callable function
        f = toolbox.compile(expr=individual)
        y_pred=[f(*point) for point in points]
        # обрежем значения
        y_pred=[min(0.99, it) for it in y_pred]
        y_pred=[max(0, it) for it in y_pred]
        
        loss = log_loss(y_true=y_true, y_pred=y_pred)
        
        return loss,
        
    # toolbox.register("evaluate", evalSymbReg, points=X_tr[filt_cols].values.tolist(), y_true=y_tr)
    toolbox.register("evaluate", evalSymbReg, points=X_val[filt_cols].values.tolist(), y_true=y_val)
    
    hof = tools.HallOfFame(1)
    
    population, logbook = algorithms.eaSimple(toolbox.population(n=100), toolbox, 
                                              cxpb=0.5, mutpb=0.2, ngen=N_GEN, halloffame=hof, verbose=False)
    
    f = toolbox.compile(hof[0])
    f_l.append(str(hof[0]))
    
    y_tr_proba = get_gen_proba(f, X_tr, filt_cols)
    y_val_proba = get_gen_proba(f, X_val, filt_cols)
    y_ts_proba = get_gen_proba(f, X_ts, filt_cols)


    average_precision_score_tr_l.append(average_precision_score(y_tr, y_tr_proba))
    average_precision_score_l.append(average_precision_score(y_ts, y_ts_proba))
    average_precision_score_val_l.append(average_precision_score(y_val, y_val_proba))
    
    _, thresh_i = get_pred_thresh(y_true = y_tr, proba=y_tr_proba, opt_metric=conf['train_eval_model']['opt_metric'],
                               thresh_space_l=np.arange(0.001, 1, 0.002))


    f1_score_l.append(f1_score(y_ts, (y_ts_proba>thresh_i)))
    f1_score_val_l.append(f1_score(y_val, (y_val_proba>thresh_i)))

    average_precision_score_tr_bert_l.append(average_precision_score(y_tr, df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])))

    average_precision_score_bert_l.append(average_precision_score(y_ts, df.query('split=="ts"')['proba_ttp'].map(lambda x: x[i])))
    average_precision_score_val_bert_l.append(average_precision_score(y_val, df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])))
    
    f1_score_bert_l.append(f1_score(y_ts, df.query('split=="ts"')['pred_ttp'].map(lambda x: x[i])))
    f1_score_bert_val_l.append(f1_score(y_val, df.query('split=="val"')['pred_ttp'].map(lambda x: x[i])))

    
end = time()

In [18]:
np.mean(average_precision_score_tr_l), np.mean(average_precision_score_val_l), np.mean(average_precision_score_val_bert_l), np.mean(f1_score_val_l), np.mean(f1_score_bert_val_l)

(0.8028497739844518,
 0.3850621966161126,
 0.37261234463714843,
 0.29300312822122376,
 0.30223213680163397)

In [19]:
np.mean(average_precision_score_l), np.mean(average_precision_score_bert_l), np.mean(f1_score_l), np.mean(f1_score_bert_l)

(0.4376898941760071,
 0.449632279754235,
 0.33539531893229196,
 0.3738654500670754)

In [20]:
comp_ts_df = pd.DataFrame({'bert':average_precision_score_bert_l, 'gen':average_precision_score_l}).assign(diff=lambda x: x['bert'] - x['gen']).sort_values(by='diff')
comp_val_df = pd.DataFrame({'bert':average_precision_score_val_bert_l, 'gen':average_precision_score_val_l}).assign(diff=lambda x: x['bert'] - x['gen']).sort_values(by='diff')
comp_tr_df = pd.DataFrame({'bert':average_precision_score_tr_bert_l, 'gen':average_precision_score_tr_l}).assign(diff=lambda x: x['bert'] - x['gen']).sort_values(by='diff')

comp_val_df.head(2)

,bert,gen,diff
65,0.583236,0.948413,-0.365176
38,0.586310,0.892857,-0.306548


In [68]:
f1_score_l[36], f1_score_bert_l[36], f1_score_val_l[36], f1_score_bert_val_l[36]

(0.2857142857142857, 0.3333333333333333, 0.4, 0.0)

In [69]:
f1_score_l[38], f1_score_bert_l[38], f1_score_val_l[38], f1_score_bert_val_l[38]

(0.5, 0.5, 0.4, 0.5)

### сравнение по классам

In [21]:
comp_val_df.merge(comp_ts_df, left_index=True, right_index=True)\
            .query('gen_x>bert_x and gen_y>bert_y')\
            .assign(diff=lambda x: x['diff_x']+ x['diff_y'])\
            .sort_values(by='diff')

,bert_x,gen_x,diff_x,bert_y,gen_y,diff_y,diff
38,0.586310,0.892857,-0.306548,0.664925,0.676316,-0.011390,-0.317938
48,0.206667,0.289683,-0.083016,0.305556,0.444444,-0.138889,-0.221905
36,0.092828,0.212564,-0.119736,0.133512,0.155719,-0.022207,-0.141943
78,0.327778,0.330409,-0.002632,0.130952,0.135965,-0.005013,-0.007644


In [26]:
comp_val_df.merge(comp_ts_df, left_index=True, right_index=True)\
            .query('gen_x>bert_x and gen_y>bert_y')\
            .assign(diff=lambda x: x['diff_x']+ x['diff_y'])\
            .sort_values(by='diff')

,bert_x,gen_x,diff_x,bert_y,gen_y,diff_y,diff
62,0.250000,0.642857,-0.392857,0.262121,0.330409,-0.068288,-0.461145
39,0.401290,0.767857,-0.366567,0.564394,0.587121,-0.022727,-0.389295
30,0.212963,0.347222,-0.134259,0.191876,0.376315,-0.184439,-0.318698
6,0.743056,0.892857,-0.149802,0.210570,0.251538,-0.040969,-0.190770
37,0.076190,0.156944,-0.080754,0.108264,0.117960,-0.009696,-0.090450
25,0.221072,0.277276,-0.056204,0.146083,0.160524,-0.014441,-0.070645
0,0.030753,0.064394,-0.033641,0.072875,0.085368,-0.012494,-0.046135
19,0.402789,0.414286,-0.011497,0.418301,0.447712,-0.029412,-0.040909
68,0.584127,0.584967,-0.000840,0.442857,0.444444,-0.001587,-0.002428
8,0.519032,0.519919,-0.000887,0.671481,0.672108,-0.000626,-0.001513


#### другой способ

In [22]:
val_gen_idx = pd.DataFrame({'bert':average_precision_score_val_bert_l, 'gen':average_precision_score_val_l}).query('gen>bert').index
val_gen_idx

Index([11, 28, 36, 38, 44, 46, 48, 49, 55, 65, 69, 78], dtype='int64')

In [23]:
ts_gen_idx = pd.DataFrame({'bert':average_precision_score_bert_l, 'gen':average_precision_score_l}).query('gen>bert').index
ts_gen_idx

Index([19, 36, 38, 48, 78], dtype='int64')

In [24]:
ts_gen_idx.intersection(val_gen_idx)

Index([36, 38, 48, 78], dtype='int64')

In [25]:
cl_l = [mlb_ttp.classes_[i_l[i]] for i in ts_gen_idx.intersection(val_gen_idx)]

In [26]:
df[df['ttp'].map(lambda x: any([it in x for it in cl_l ]))].explode('ttp')[['split', 'ttp']].groupby(['ttp', 'split']).size()

ttp        split
T1132.002  tr       12
           ts        3
           val       3
T1134      tr       17
           ts        4
           val       4
T1539      tr       14
           ts        3
           val       3
T1608.004  tr       11
           ts        2
           val       3
dtype: int64

In [33]:
df[df['ttp'].map(lambda x: any([it in x for it in cl_l ]))].explode('ttp')[['split', 'ttp']].groupby(['ttp', 'split']).size()

ttp        split
T1003      tr       11
           ts        3
           val       2
T1025      tr       19
           ts        5
           val       4
T1090.002  tr       15
           ts        4
           val       4
T1114.001  tr       10
           ts        3
           val       3
T1132.002  tr       12
           ts        3
           val       3
T1134      tr       17
           ts        4
           val       4
T1561.002  tr       11
           ts        3
           val       2
T1584.004  tr       12
           ts        3
           val       3
dtype: int64

In [ ]:
- сделаем при разных сидах для всех классов, где у нас больше 19

## i = 38

In [46]:
i = 38
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


,feat,
0,tak_0,ARG0
1,ttp_i,ARG1


In [47]:
f_l[i]

'mul(ARG0, ARG1)'

In [48]:
mlb.classes_[0]

'defense-evasion'

In [49]:
df.loc[df.ttp.map(lambda x: 'T1134' in x), 'labels'].unique()

array(["['defense-evasion', 'privilege-escalation']"], dtype=object)

## i = 36

In [51]:
i = 36
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


,feat,
0,tak_4,ARG0
1,tak_12,ARG1
2,ttp_i,ARG2


In [52]:
mlb.classes_[4]

'command-and-control'

In [53]:
df.loc[df.ttp.map(lambda x: 'T1132.002' in x), 'labels'].unique()

array(["['command-and-control']"], dtype=object)

In [54]:
mlb_ttp.classes_[i_l[i]]

'T1132.002'

In [56]:
i, i_l[i]

(36, 102)

In [36]:
f_l[i]

'mul(ARG0, ARG2)'

In [37]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

(0.21256410256410257,
 0.09282781936730257,
 0.155718954248366,
 0.1335118425635667)

In [40]:
df.query('split=="val"')

,sentence,labels,ttp,split,target_ttp,target,proba_labels,pred_labels,proba_ttp,pred_ttp
4,Running code in the context of another process...,"['defense-evasion', 'privilege-escalation']",[rare],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.9697772264480591, 0.9689617156982422, 0.074...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.6878567521416699e-07, 3.712784746312536e-05...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,"Adversaries may also create ""hidden"" scheduled...","['execution', 'persistence', 'privilege-escala...",[T1053.005],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.5470336079597473, 0.1475258767604828, 0.224...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.0024490620708093e-05, 6.513068910862785e-06...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,Adversaries may attach filters to a network so...,"['defense-evasion', 'persistence', 'command-an...",[rare],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.19366611540317535, 0.18692217767238617, 0.4...","[0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[1.9804978364845738e-05, 1.6051106285885908e-0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
24,"Adversaries may store data in ""fileless"" forma...",['defense-evasion'],[T1027.011],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.7150190472602844, 0.004406386520713568, 0.0...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4.818417437491007e-06, 1.4215402188710868e-07...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
27,Some forms of fileless storage activity may in...,['defense-evasion'],[T1027.011],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.4914970397949219, 0.09078268706798553, 0.01...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.652781378425061e-07, 1.623723875354699e-07,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...
30701,SEABORGIUM and TA453 actors register consumer ...,['resource-development'],[T1585.002],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.0006819515256211162, 0.00019326090114191175...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]","[1.24427586456477e-07, 2.1598939383693505e-06,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
30704,SEABORGIUM and TA453 actors use compromised cr...,"['defense-evasion', 'persistence', 'privilege-...",[T1078],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.4315950274467468, 0.4902055859565735, 0.007...","[1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]","[3.2848703312993166e-07, 5.2700299420394003e-0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
30705,SEABORGIUM actors use malicious links embedded...,['initial-access'],[T1566.001],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.020760027691721916, 0.0019938850309699774, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[4.297745590520208e-07, 1.2072470099155908e-06...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
30706,SEABORGIUM actors send spear-phishing emails w...,['initial-access'],[T1566.002],val,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[0.014318901114165783, 0.001014973153360188, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[1.6324150919899694e-08, 3.262783110358214e-08...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [50]:
i_l[i]

104

In [41]:
y_val = df.query('split=="val"')['target_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][8], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

3

(0.015156758538293577, 0.09282781936730257)

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==1][['gen','bert']]

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>